# RawTreeWriter: 生データツリー

`ReadData` の直後につなぎ、**検出器間で同期した生データ**を 1 イベント 1 行の TTree `RawTreeWriter/raw` に書き出します。
これを使えば、閾値などを変えた再解析や、ゲインマップ・TPC の較正係数・ドリフト速度の算出を、
生データ(raw ファイル)を読み直さずに行えます。各モジュールの解説ノートブックの図もここから描いています。

| ブランチ | 型 | 中身 |
|---|---|---|
| `event_id`, `sevt_id` | uint | トリガ番号・通し番号(`single_hit` と同じ) |
| `loopback` | int | イベント同期で読み飛ばしが起きたら 1 |
| `anode_time` | vector<uint> | アノードボードのヘッダ時刻(0.1 ms 単位、DAQ 開始から) |
| `hit_a_strip`, `hit_a_clock` | vector<short> | アノードの全ヒット(ストリップ 0–383、クロック) |
| `hit_c_strip`, `hit_c_clock` | vector<short> | カソードの全ヒット |
| `fadc_ch` | vector<short> | 波形を保存した FADC ch(0–11 アノード = 4×ボード + ch、12–23 カソード) |
| `fadc_wave` | vector<short> | 上の ch ごとに 512 サンプルの波形(カソードは `1023 − ADC` に反転済み = CalcFADC が受け取る形) |
| `ha_adc`, `ha_err` | vector<short>, vector<char> | HA 18 台 × 24 ch の ADC(12 bit)とエラービット(並びは HA_CenterGravity と同じ: PMT ごとに 4 ch) |

別ツリー `RawTreeWriter/run_info` にボード ID の一覧が入っています。

### パラメータ

| パラメータ | 既定 | 意味 |
|---|---|---|
| `save_fadc`, `save_ha` | `True` | 波形・HA を保存するか |
| `save_threshold` | 0(オフ) | > 0 なら、ベースラインからの最大振幅がこれを超えた ch の波形だけ保存 |
| `base_window` | 30 | 上の判定に使うベースラインのサンプル数 |

データのないボード(波形がすべて 0)は保存しません。`SaveFile` の `compression` を 505(zstd)にすると、
1 期間(約 82 万イベント)で約 5 GB です(生データは 17 GB)。

In [1]:
import sys
from pathlib import Path
here = Path.cwd().resolve()
sys.path.insert(0, str(next(p for p in [here, *here.parents] if (p / "tutorial_utils.py").exists())))
import tutorial_utils as tu

import numpy as np
import matplotlib.pyplot as plt
ROOT = tu.setup_pyroot()

## 1. データのダウンロード

このチュートリアルでは 20190701 の較正データ **Per02**(線源を TPC の上方 z = −1626 mm に置いた測定)を使います。
生データ(約 17 GB)とパラメータはリポジトリに含まれていないので、研究室サーバーから手元にコピーします。

```bash
mkdir -p ~/smile2p_data
rsync -av --progress <user>@ramune:/home/onoda/hikaru3_analysis2/sm2+_cal_rawdata/Per02_ETCC ~/smile2p_data/
rsync -av --progress <user>@ramune:/home/onoda/hikaru3_analysis2/sm2+_cal_rawdata/param ~/smile2p_data/
```

(`<user>` は自分のアカウント名)

- 途中で切れても同じコマンドで続きからコピーされます。
- `~/smile2p_data` 以外に置いた場合は、環境変数 `SMILE2P_DATA` にその場所を設定してください。
- サーバー上で実行する場合はコピーしなくても動きます(サーバー上の場所をそのまま使います)。
  ただし実行時間の大半は生データの読み込みなので、**手元のディスクにコピーした方が大幅に速く**なります
  (ネットワーク越しの場所から読むと約 10 倍遅い)。

In [2]:
tu.print_download_instructions("Per02")
DATA_TOP = tu.check_data("Per02")

# 生データ(Per02)とパラメータを ~/smile2p_data にコピーする
mkdir -p ~/smile2p_data
rsync -av --progress <user>@ramune:/home/onoda/hikaru3_analysis2/sm2+_cal_rawdata/Per02_ETCC ~/smile2p_data/
rsync -av --progress <user>@ramune:/home/onoda/hikaru3_analysis2/sm2+_cal_rawdata/param ~/smile2p_data/
data OK: /home/onoda/hikaru3_analysis2/sm2+_cal_rawdata  (Per02: 554 raw files)


## 2. 生データツリーを作る

In [3]:
# ReadData -> RawTreeWriter の出力(tutorials/cache/ に保存し、他のモジュールのノートブックと共有)
RAW = tu.get_raw("Per02", 5000)
INFO = tu.raw_info(RAW)
INFO

Per02_raw_5000.root は既にあるので再実行しません(overwrite=True で作り直し)


{'anode_ids': [1, 2, 3],
 'cathode_ids': [1, 2, 3],
 'ha_ids': [1, 2, 3, 7, 8, 9, 13, 14, 15, 19, 20, 21, 25, 26, 27, 31, 32, 33]}

## 3. 中身

In [4]:
f = ROOT.TFile.Open(str(RAW))
t = f.Get("RawTreeWriter/raw")
n = t.GetEntries()
print(f"{n} events, file {RAW.stat().st_size / n / 1e3:.1f} kB/event")
for b in t.GetListOfBranches():
    print(f"  {b.GetName():12s} {b.GetZipBytes() / n:7.0f} B/event (compressed)")

5000 events, file 6.6 kB/event
  event_id           3 B/event (compressed)
  sevt_id            2 B/event (compressed)
  loopback           0 B/event (compressed)
  anode_time         9 B/event (compressed)
  hit_a_strip      202 B/event (compressed)
  hit_a_clock      354 B/event (compressed)
  hit_c_strip      231 B/event (compressed)
  hit_c_clock      370 B/event (compressed)
  fadc_ch            5 B/event (compressed)
  fadc_wave       5268 B/event (compressed)
  ha_adc            95 B/event (compressed)
  ha_err            63 B/event (compressed)


1 イベントの中身の例:

In [5]:
e = next(tu.iter_raw(RAW, ["sevt_id", "anode_time", "hit_a_strip", "hit_a_clock", "fadc_ch"]))
print("sevt_id", e["sevt_id"], " anode_time", e["anode_time"])
print("anode hits:", len(e["hit_a_strip"]), " first:", list(zip(e["hit_a_strip"][:5], e["hit_a_clock"][:5])))
print("FADC channels with waveform:", e["fadc_ch"])

sevt_id 0  anode_time [1619705216      23618 1665711824]
anode hits: 1493  first: [(125, 526), (127, 526), (123, 527), (124, 527), (125, 527)]
FADC channels with waveform: [ 0  1  2  3  4  5  6  7  8  9 10 11 16 17 18 19 20 21 22 23]
